# SAC Hyperparameter Tuning

Este notebook implementa hyperparameter tuning para el algoritmo SAC (Soft Actor-Critic) utilizando Ray Tune. 

El código reutiliza las funciones y configuraciones de `run.py` para mantener consistencia con el pipeline de entrenamiento existente.

## Objetivos:
- Optimizar hiperparámetros del algoritmo SAC
- Encontrar las mejores configuraciones para diferentes mapas
- Comparar rendimiento entre diferentes configuraciones

## Estrategia de búsqueda:
- Utilizaremos diferentes algoritmos de búsqueda (grid search, random search, Bayesian optimization)
- Optimizaremos métricas clave como episode_return_mean
- Implementaremos early stopping para eficiencia computacional

In [22]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import yaml
import json
from typing import Dict, Any, Optional, List

# Ray imports
import ray
from ray import tune
from ray.tune import Tuner, TuneConfig
from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.bayesopt import BayesOptSearch
from ray.tune.search.hyperopt import HyperOptSearch
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
from ray.tune.stopper import CombinedStopper, TrialPlateauStopper

# RLLib imports
from ray.rllib.algorithms.sac import SACConfig
from ray.rllib.policy.policy import PolicySpec
from ray.rllib.algorithms.algorithm import Algorithm

# Importar las funciones de run.py (reutilizando código existente)
sys.path.append('.')
from lib.utils import (
    load_config,
    init_ray,
    get_logger,
    suppress_warnings,
    get_experiment_path,
    get_best_checkpoint,
    get_reward_class,
    setup_experiment_config,
    find_experiment
)
from lib.callbacks import SaveConfig, MultipleAgentCallbacks

# Configurar logging
suppress_warnings()
logger = get_logger(__name__)

print("✅ Imports completados exitosamente")

✅ Imports completados exitosamente


In [24]:
# ================================
# CONFIGURACIÓN INICIAL
# ================================

# Configuración de paths
CONFIG_PATH = Path('configs/experiments.yaml')
BASE_STORAGE_PATH = Path('../models/hyperparameter_tuning')

# Crear directorio de almacenamiento si no existe
BASE_STORAGE_PATH.mkdir(parents=True, exist_ok=True)

# Configuraciones para el tuning
TUNING_CONFIG = {
    'base_experiment': 'oval_small_SAC_Shared_Policy',  # Experimento base a usar como template
    'num_samples': 20,  # Número de configuraciones a probar
    'max_concurrent_trials': 3,  # Número máximo de trials concurrentes
    'grace_period': 10,  # Período de gracia para early stopping
    'max_timesteps': 1_000_000,  # Timesteps máximos por trial (reducido para tuning)
    'metric': 'env_runners/episode_return_mean',  # Métrica a optimizar
    'mode': 'max',  # Maximizar la métrica
}

print("🔧 Configuración inicial completada")
print(f"📁 Directorio de almacenamiento: {BASE_STORAGE_PATH}")
print(f"🎯 Experimento base: {TUNING_CONFIG['base_experiment']}")
print(f"🔢 Número de samples: {TUNING_CONFIG['num_samples']}")
print(f"⏱️ Max timesteps por trial: {TUNING_CONFIG['max_timesteps']:,}")


# ================================
# CLASES HELPER
# ================================

class TimestepsStopper:
    """Custom stopper que para el entrenamiento después de ciertos timesteps."""
    
    def __init__(self, max_timesteps):
        self.max_timesteps = max_timesteps
    
    def __call__(self, trial_id, result):
        """Para el trial si timesteps_total excede max_timesteps."""
        return result.get("timesteps_total", 0) >= self.max_timesteps
    
    def stop_all(self):
        """No para todos los experimentos."""
        return False


def create_env_from_config(config, render_mode=None):
    """Crea un environment basado en la configuración (reutilizado de run.py)."""
    suppress_warnings()
    env_config = config['env'].copy()
    
    if render_mode:
        env_config["render_mode"] = render_mode
    
    reward_class = get_reward_class(config)
    return reward_class(env_config=env_config), env_config


print("🛠️ Clases helper definidas exitosamente")

🔧 Configuración inicial completada
📁 Directorio de almacenamiento: ../models/hyperparameter_tuning
🎯 Experimento base: oval_small_SAC_Shared_Policy
🔢 Número de samples: 20
⏱️ Max timesteps por trial: 1,000,000
🛠️ Clases helper definidas exitosamente


In [25]:
# ================================
# CARGAR CONFIGURACIÓN BASE
# ================================

# Cargar configuración desde experiments.yaml
config_path = CONFIG_PATH.resolve()
config_dir = config_path.parent
config_data = load_config(config_path)
experiments = config_data.get('experiments', [])

# Encontrar el experimento base
base_experiment = find_experiment(experiments, TUNING_CONFIG['base_experiment'])
base_config = setup_experiment_config(base_experiment, config_dir)

print("📋 Configuración base cargada:")
print(f"  - Nombre: {base_config['name']}")
print(f"  - Algoritmo: {base_config['training']['algorithm']}")
print(f"  - Mapa: {base_config['env']['map']}")
print(f"  - Shared Policy: {base_config['training']['shared_policy']}")
print(f"  - Reward Function: {base_config['training']['reward_function']}")

# Mostrar configuración SAC actual
print("\n🔍 Configuración SAC actual:")
sac_params = base_config.get('sac_params', {})
for key, value in sac_params.items():
    print(f"  - {key}: {value}")

# Inicializar Ray
init_ray()
print("\n🚀 Ray inicializado correctamente")

2025-07-06 10:53:23,830	INFO worker.py:1747 -- Calling ray.init() again after it has already been called.


📋 Configuración base cargada:
  - Nombre: oval_small_SAC_Shared_Policy
  - Algoritmo: SAC
  - Mapa: oval_small
  - Shared Policy: True
  - Reward Function: KohondaMultiAgentF110Env

🔍 Configuración SAC actual:
  - tau: 0.005
  - target_entropy: auto
  - alpha_lr: 0.0003
  - environment: {'normalize_actions': True}
  - replay_buffer_config: {'type': 'MultiAgentPrioritizedReplayBuffer'}

🚀 Ray inicializado correctamente


In [26]:
# ================================
# DEFINIR ESPACIO DE BÚSQUEDA
# ================================

# Definir diferentes espacios de búsqueda para hiperparámetros SAC
search_spaces = {
    'basic': {
        # Hiperparámetros básicos de SAC
        'lr': tune.loguniform(1e-5, 1e-3),  # Learning rate
        'tau': tune.uniform(0.001, 0.01),   # Soft update coefficient
        'gamma': tune.uniform(0.95, 0.999), # Discount factor
        'alpha_lr': tune.loguniform(1e-5, 1e-3),  # Alpha learning rate
        'train_batch_size': tune.choice([256, 512, 1024, 2048]),
        'replay_buffer_size': tune.choice([100000, 500000, 1000000]),
    },
    
    'extended': {
        # Hiperparámetros extendidos
        'lr': tune.loguniform(1e-5, 1e-3),
        'tau': tune.uniform(0.001, 0.01),
        'gamma': tune.uniform(0.95, 0.999),
        'alpha_lr': tune.loguniform(1e-5, 1e-3),
        'train_batch_size': tune.choice([256, 512, 1024, 2048]),
        'replay_buffer_size': tune.choice([100000, 500000, 1000000]),
        'target_network_update_freq': tune.choice([1, 2, 4]),
        'num_steps_sampled_before_learning_starts': tune.choice([1000, 5000, 10000]),
        'twin_q': tune.choice([True, False]),
        'clip_actions': tune.choice([True, False]),
    },
    
    'grid': {
        # Búsqueda en grid (más controlada)
        'lr': tune.grid_search([1e-4, 5e-4, 1e-3]),
        'tau': tune.grid_search([0.001, 0.005, 0.01]),
        'gamma': tune.grid_search([0.99, 0.995, 0.999]),
        'alpha_lr': tune.grid_search([1e-4, 5e-4, 1e-3]),
        'train_batch_size': tune.grid_search([512, 1024, 2048]),
    }
}

# Seleccionar espacio de búsqueda
SELECTED_SEARCH_SPACE = 'basic'  # Cambiar a 'extended' o 'grid' según necesidad
current_search_space = search_spaces[SELECTED_SEARCH_SPACE]

print(f"🔍 Espacio de búsqueda seleccionado: {SELECTED_SEARCH_SPACE}")
print(f"📊 Parámetros a optimizar: {list(current_search_space.keys())}")

# Calcular número estimado de configuraciones
if SELECTED_SEARCH_SPACE == 'grid':
    # Para grid search, calculamos el producto cartesiano
    total_combinations = 1
    for param_name, param_config in current_search_space.items():
        if hasattr(param_config, 'categories'):
            total_combinations *= len(param_config.categories)
    print(f"🔢 Configuraciones totales (grid): {total_combinations}")
else:
    print(f"🔢 Configuraciones a muestrear: {TUNING_CONFIG['num_samples']}")

print("\n📝 Resumen del espacio de búsqueda:")
for param_name, param_config in current_search_space.items():
    print(f"  - {param_name}: {param_config}")

🔍 Espacio de búsqueda seleccionado: basic
📊 Parámetros a optimizar: ['lr', 'tau', 'gamma', 'alpha_lr', 'train_batch_size', 'replay_buffer_size']
🔢 Configuraciones a muestrear: 20

📝 Resumen del espacio de búsqueda:
  - lr: <ray.tune.search.sample.Float object at 0x78dc92f5e840>
  - tau: <ray.tune.search.sample.Float object at 0x78dc92f5e5d0>
  - gamma: <ray.tune.search.sample.Float object at 0x78dc92f5f0e0>
  - alpha_lr: <ray.tune.search.sample.Float object at 0x78dc92dc9a30>
  - train_batch_size: <ray.tune.search.sample.Categorical object at 0x78dc92dc9a00>
  - replay_buffer_size: <ray.tune.search.sample.Categorical object at 0x78dc92dcaf00>


In [27]:
# ================================
# FUNCIÓN DE ENTRENAMIENTO TUNEABLE
# ================================

def trainable_sac(config_tune, base_config=None):
    """
    Función de entrenamiento que será optimizada por Ray Tune.
    
    Args:
        config_tune: Diccionario con hiperparámetros a optimizar
        base_config: Configuración base del experimento
    """
    suppress_warnings()
    
    # Crear una copia de la configuración base
    config = base_config.copy()
    
    # Actualizar los hiperparámetros SAC con los valores de tuning
    sac_params = config.get('sac_params', {}).copy()
    
    # Mapear hiperparámetros de tune a configuración SAC
    param_mapping = {
        'lr': 'lr',
        'tau': 'tau', 
        'gamma': 'gamma',
        'alpha_lr': 'alpha_lr',
        'train_batch_size': 'train_batch_size',
        'replay_buffer_size': 'replay_buffer_size',
        'target_network_update_freq': 'target_network_update_freq',
        'num_steps_sampled_before_learning_starts': 'num_steps_sampled_before_learning_starts',
        'twin_q': 'twin_q',
        'clip_actions': 'clip_actions',
    }
    
    # Actualizar parámetros SAC
    for tune_param, sac_param in param_mapping.items():
        if tune_param in config_tune:
            sac_params[sac_param] = config_tune[tune_param]
    
    config['sac_params'] = sac_params
    
    # Reducir timesteps para tuning (más eficiente)
    config['training']['timesteps_total'] = TUNING_CONFIG['max_timesteps']
    
    # Crear environment temporal para obtener espacios de observación y acción
    temp_env, env_config = create_env_from_config(config)
    
    # Configurar políticas (reutilizando lógica de run.py)
    shared_policy = config['training'].get('shared_policy', True)
    
    if shared_policy:
        shared_policy_name = "shared_policy"
        policies = {
            shared_policy_name: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {})
        }
        policy_mapping_fn = lambda agent_id, *args, **kwargs: shared_policy_name
    else:
        policies = {
            agent: PolicySpec(None, temp_env.observation_space, temp_env.action_space, {})
            for agent in temp_env.agents
        }
        policy_mapping_fn = lambda agent_id, *args, **kwargs: agent_id
    
    temp_env.close()
    
    # Configurar SAC (reutilizando lógica de run.py)
    env_kwargs = sac_params.get('environment', {})
    sac_config_clean = sac_params.copy()
    if 'environment' in sac_config_clean:
        del sac_config_clean['environment']
    
    sac_config = (
        SACConfig()
        .environment(get_reward_class(config), env_config=env_config, **env_kwargs)
        .framework("torch")
        .api_stack(
            enable_rl_module_and_learner=False,
            enable_env_runner_and_connector_v2=False,
        )
        .callbacks(MultipleAgentCallbacks)
        .multi_agent(
            policies=policies,
            policy_mapping_fn=policy_mapping_fn,
        )
        .evaluation(
            evaluation_interval=config["training"]["eval_interval"],
            evaluation_num_env_runners=1,
            evaluation_config={"seed": 42},
        )
        .env_runners(
            num_env_runners=6,  # Reducido para tuning
            num_envs_per_env_runner=4,  # Reducido para tuning
            gym_env_vectorize_mode="ASYNC"
        )
        .debugging(seed=42)
    )
    
    # Aplicar configuración de entrenamiento
    sac_config.training(**sac_config_clean)
    
    # Crear y entrenar el algoritmo
    algo = sac_config.build()
    
    # Stopper personalizado
    timesteps_stopper = TimestepsStopper(max_timesteps=TUNING_CONFIG['max_timesteps'])
    
    # Loop de entrenamiento
    iteration = 0
    while True:
        result = algo.train()
        iteration += 1
        
        # Reportar métricas a Tune
        tune.report(
            iteration=iteration,
            timesteps_total=result.get("timesteps_total", 0),
            env_runners_episode_return_mean=result.get("env_runners/episode_return_mean", 0),
            env_runners_episode_len_mean=result.get("env_runners/episode_len_mean", 0),
            learner_alpha=result.get("learner/alpha", 0),
            learner_critic_loss=result.get("learner/critic_loss", 0),
            learner_policy_loss=result.get("learner/policy_loss", 0),
        )
        
        # Verificar condición de parada
        if timesteps_stopper(None, result):
            break
    
    algo.stop()

print("🎯 Función de entrenamiento tuneable definida exitosamente")
print("📊 Métricas que se reportarán:")
print("  - env_runners/episode_return_mean (principal)")
print("  - env_runners/episode_len_mean")
print("  - learner/alpha")
print("  - learner/critic_loss")
print("  - learner/policy_loss")

🎯 Función de entrenamiento tuneable definida exitosamente
📊 Métricas que se reportarán:
  - env_runners/episode_return_mean (principal)
  - env_runners/episode_len_mean
  - learner/alpha
  - learner/critic_loss
  - learner/policy_loss


In [28]:
# ================================
# CONFIGURAR ALGORITMOS DE BÚSQUEDA Y SCHEDULERS
# ================================
from ray.tune.search.bayesopt import BayesOptSearch

# Configurar diferentes algoritmos de búsqueda
search_algorithms = {
    'random': None,  # Búsqueda aleatoria (por defecto)
    
    'bayesian': BayesOptSearch(
        metric=TUNING_CONFIG['metric'],
        mode=TUNING_CONFIG['mode'],
        random_search_steps=4,  # Pasos de búsqueda aleatoria inicial
    ),
    
    'hyperopt': HyperOptSearch(
        metric=TUNING_CONFIG['metric'],
        mode=TUNING_CONFIG['mode'],
        n_initial_points=4,  # Puntos iniciales aleatorios
    ),
}

# Configurar schedulers para early stopping
schedulers = {
    'asha': ASHAScheduler(
        metric=TUNING_CONFIG['metric'],
        mode=TUNING_CONFIG['mode'],
        max_t=TUNING_CONFIG['max_timesteps'],
        grace_period=TUNING_CONFIG['grace_period'],
        reduction_factor=2,
    ),
    
    'pbt': PopulationBasedTraining(
        metric=TUNING_CONFIG['metric'],
        mode=TUNING_CONFIG['mode'],
        perturbation_interval=10,
        hyperparam_mutations={
            'lr': lambda: np.random.uniform(1e-5, 1e-3),
            'tau': lambda: np.random.uniform(0.001, 0.01),
            'gamma': lambda: np.random.uniform(0.95, 0.999),
        },
    ),
    
    'plateau': TrialPlateauStopper(
        metric=TUNING_CONFIG['metric'],
        std=5.0,
        num_results=10,
        mode=TUNING_CONFIG['mode'],
    ),
}

# Configuración de experimentos de tuning
tuning_experiments = {
    'sac_random_search': {
        'search_algorithm': 'random',
        'scheduler': 'asha',
        'description': 'Búsqueda aleatoria con ASHA scheduler',
    },
    
    'sac_bayesian_optimization': {
        'search_algorithm': 'bayesian',
        'scheduler': 'asha',
        'description': 'Optimización Bayesiana con ASHA scheduler',
    },
    
    'sac_hyperopt': {
        'search_algorithm': 'hyperopt',
        'scheduler': 'asha',
        'description': 'HyperOpt con ASHA scheduler',
    },
    
    'sac_pbt': {
        'search_algorithm': 'random',
        'scheduler': 'pbt',
        'description': 'Population Based Training',
    },
}

# Seleccionar configuración de experimento
SELECTED_EXPERIMENT = 'sac_random_search'  # Cambiar según necesidad
current_experiment = tuning_experiments[SELECTED_EXPERIMENT]

print(f"🔬 Experimento seleccionado: {SELECTED_EXPERIMENT}")
print(f"📝 Descripción: {current_experiment['description']}")
print(f"🔍 Algoritmo de búsqueda: {current_experiment['search_algorithm']}")
print(f"📅 Scheduler: {current_experiment['scheduler']}")

# Verificar límites de concurrencia
search_algo = search_algorithms[current_experiment['search_algorithm']]
scheduler = schedulers[current_experiment['scheduler']]

if search_algo is not None:
    # Aplicar límite de concurrencia para algoritmos de búsqueda avanzados
    search_algo = ConcurrencyLimiter(search_algo, max_concurrent=TUNING_CONFIG['max_concurrent_trials'])
    print(f"⚡ Límite de concurrencia aplicado: {TUNING_CONFIG['max_concurrent_trials']} trials")

print("\n✅ Configuración de búsqueda completada")

AttributeError: module 'bayes_opt' has no attribute 'UtilityFunction'

In [ ]:
# ================================
# EJECUTAR HYPERPARAMETER TUNING
# ================================

# Preparar nombre del experimento
experiment_name = f"SAC_Tuning_{SELECTED_EXPERIMENT}_{base_config['env']['map']}"
storage_path = str(BASE_STORAGE_PATH.resolve())

print(f"🚀 Iniciando hyperparameter tuning...")
print(f"📊 Experimento: {experiment_name}")
print(f"📁 Almacenamiento: {storage_path}")
print(f"⏱️ Timesteps por trial: {TUNING_CONFIG['max_timesteps']:,}")
print(f"🔢 Número de trials: {TUNING_CONFIG['num_samples']}")
print(f"🎯 Métrica objetivo: {TUNING_CONFIG['metric']} ({TUNING_CONFIG['mode']})")

# Crear función trainable con configuración base
trainable_with_config = tune.with_parameters(trainable_sac, base_config=base_config)

# Configurar el tuner
tuner = Tuner(
    trainable_with_config,
    param_space=current_search_space,
    tune_config=TuneConfig(
        metric=TUNING_CONFIG['metric'],
        mode=TUNING_CONFIG['mode'],
        search_alg=search_algo,
        scheduler=scheduler,
        num_samples=TUNING_CONFIG['num_samples'],
        max_concurrent_trials=TUNING_CONFIG['max_concurrent_trials'],
    ),
    run_config=ray.train.RunConfig(
        name=experiment_name,
        storage_path=storage_path,
        checkpoint_config=ray.train.CheckpointConfig(
            num_to_keep=3,
            checkpoint_score_attribute=TUNING_CONFIG['metric'],
            checkpoint_score_order=TUNING_CONFIG['mode'],
        ),
        failure_config=ray.train.FailureConfig(max_failures=3),
        verbose=2,
    ),
)

print("\n🎯 Configuración del tuner completada")
print("⚠️  ADVERTENCIA: El tuning puede tomar varias horas dependiendo de la configuración")
print("📊 Puedes monitorear el progreso en Ray Dashboard (http://localhost:8265)")

# Ejecutar el tuning
print("\n🔥 Iniciando tuning...")
try:
    results = tuner.fit()
    print("✅ Tuning completado exitosamente!")
    
    # Guardar referencia de resultados para análisis posterior
    tuning_results = results
    
except Exception as e:
    print(f"❌ Error durante el tuning: {e}")
    raise e

In [ ]:
# ================================
# ANÁLISIS DE RESULTADOS
# ================================

def analyze_tuning_results(results):
    """Analiza los resultados del hyperparameter tuning."""
    
    print("📊 ANÁLISIS DE RESULTADOS DE TUNING")
    print("=" * 50)
    
    # Obtener el mejor resultado
    best_result = results.get_best_result(metric=TUNING_CONFIG['metric'], mode=TUNING_CONFIG['mode'])
    
    print(f"🏆 MEJOR CONFIGURACIÓN:")
    print(f"📈 {TUNING_CONFIG['metric']}: {best_result.metrics[TUNING_CONFIG['metric']]:.4f}")
    print(f"⏱️ Timesteps: {best_result.metrics.get('timesteps_total', 0):,}")
    print(f"🔄 Iteraciones: {best_result.metrics.get('iteration', 0)}")
    
    print(f"\n🎯 HIPERPARÁMETROS ÓPTIMOS:")
    for param, value in best_result.config.items():
        print(f"  - {param}: {value}")
    
    # Convertir todos los resultados a DataFrame para análisis
    df_results = results.get_dataframe()
    
    print(f"\n📊 ESTADÍSTICAS GENERALES:")
    print(f"  - Total de trials: {len(df_results)}")
    print(f"  - Trials completados: {len(df_results[df_results['trial_status'] == 'TERMINATED'])}")
    print(f"  - Trials fallidos: {len(df_results[df_results['trial_status'] == 'ERROR'])}")
    
    # Estadísticas de la métrica objetivo
    metric_col = f"last_{TUNING_CONFIG['metric'].replace('/', '_')}"
    if metric_col in df_results.columns:
        metric_stats = df_results[metric_col].describe()
        print(f"\n📈 ESTADÍSTICAS DE {TUNING_CONFIG['metric']}:")
        print(f"  - Media: {metric_stats['mean']:.4f}")
        print(f"  - Std: {metric_stats['std']:.4f}")
        print(f"  - Min: {metric_stats['min']:.4f}")
        print(f"  - Max: {metric_stats['max']:.4f}")
    
    return df_results, best_result

# Ejecutar análisis si hay resultados disponibles
if 'tuning_results' in locals():
    df_results, best_result = analyze_tuning_results(tuning_results)
    
    # Guardar resultados
    results_file = BASE_STORAGE_PATH / f"{experiment_name}_results.csv"
    df_results.to_csv(results_file, index=False)
    print(f"\n💾 Resultados guardados en: {results_file}")
    
    # Guardar mejor configuración
    best_config_file = BASE_STORAGE_PATH / f"{experiment_name}_best_config.json"
    with open(best_config_file, 'w') as f:
        json.dump(best_result.config, f, indent=2)
    print(f"💾 Mejor configuración guardada en: {best_config_file}")
    
else:
    print("⚠️  No hay resultados de tuning disponibles.")
    print("   Ejecuta primero la celda de tuning para generar resultados.")

In [ ]:
# ================================
# VISUALIZACIÓN DE RESULTADOS
# ================================

def create_tuning_visualizations(df_results, best_result):
    """Crea visualizaciones de los resultados del tuning."""
    
    plt.style.use('seaborn-v0_8')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(f'Análisis de Hyperparameter Tuning - {experiment_name}', fontsize=16, fontweight='bold')
    
    # 1. Distribución de la métrica objetivo
    metric_col = f"last_{TUNING_CONFIG['metric'].replace('/', '_')}"
    if metric_col in df_results.columns:
        axes[0, 0].hist(df_results[metric_col].dropna(), bins=20, alpha=0.7, color='skyblue', edgecolor='black')
        axes[0, 0].axvline(best_result.metrics[TUNING_CONFIG['metric']], color='red', linestyle='--', 
                          label=f'Mejor: {best_result.metrics[TUNING_CONFIG["metric"]]:.4f}')
        axes[0, 0].set_title(f'Distribución de {TUNING_CONFIG["metric"]}')
        axes[0, 0].set_xlabel('Valor')
        axes[0, 0].set_ylabel('Frecuencia')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Progreso de trials (si hay columna de timesteps)
    timesteps_col = 'last_timesteps_total'
    if timesteps_col in df_results.columns:
        axes[0, 1].scatter(df_results[timesteps_col], df_results[metric_col], alpha=0.6, color='green')
        axes[0, 1].set_title('Rendimiento vs Timesteps')
        axes[0, 1].set_xlabel('Timesteps')
        axes[0, 1].set_ylabel(TUNING_CONFIG['metric'])
        axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Comparación de hiperparámetros importantes
    important_params = ['config/lr', 'config/tau', 'config/gamma', 'config/train_batch_size']
    available_params = [p for p in important_params if p in df_results.columns]
    
    if available_params:
        param = available_params[0]  # Usar el primer parámetro disponible
        axes[1, 0].scatter(df_results[param], df_results[metric_col], alpha=0.6, color='orange')
        axes[1, 0].set_title(f'Rendimiento vs {param.split("/")[-1]}')
        axes[1, 0].set_xlabel(param.split("/")[-1])
        axes[1, 0].set_ylabel(TUNING_CONFIG['metric'])
        axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Top 5 configuraciones
    top_5 = df_results.nlargest(5, metric_col) if TUNING_CONFIG['mode'] == 'max' else df_results.nsmallest(5, metric_col)
    if not top_5.empty:
        axes[1, 1].barh(range(len(top_5)), top_5[metric_col], color='purple', alpha=0.7)
        axes[1, 1].set_title('Top 5 Configuraciones')
        axes[1, 1].set_xlabel(TUNING_CONFIG['metric'])
        axes[1, 1].set_ylabel('Ranking')
        axes[1, 1].set_yticks(range(len(top_5)))
        axes[1, 1].set_yticklabels([f'#{i+1}' for i in range(len(top_5))])
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Guardar visualización
    viz_file = BASE_STORAGE_PATH / f"{experiment_name}_visualization.png"
    plt.savefig(viz_file, dpi=300, bbox_inches='tight')
    print(f"📊 Visualización guardada en: {viz_file}")
    
    plt.show()
    
    return fig

def create_hyperparameter_heatmap(df_results):
    """Crea un heatmap de correlación entre hiperparámetros y rendimiento."""
    
    # Filtrar columnas de configuración
    config_cols = [col for col in df_results.columns if col.startswith('config/')]
    metric_col = f"last_{TUNING_CONFIG['metric'].replace('/', '_')}"
    
    if config_cols and metric_col in df_results.columns:
        # Crear DataFrame con solo hiperparámetros numéricos
        df_numeric = df_results[config_cols + [metric_col]].select_dtypes(include=[np.number])
        
        if not df_numeric.empty:
            plt.figure(figsize=(10, 8))
            correlation_matrix = df_numeric.corr()
            
            # Crear heatmap
            sns.heatmap(correlation_matrix, 
                       annot=True, 
                       cmap='coolwarm', 
                       center=0,
                       square=True,
                       fmt='.3f')
            
            plt.title(f'Correlación entre Hiperparámetros y {TUNING_CONFIG["metric"]}')
            plt.tight_layout()
            
            # Guardar heatmap
            heatmap_file = BASE_STORAGE_PATH / f"{experiment_name}_heatmap.png"
            plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
            print(f"🔥 Heatmap guardado en: {heatmap_file}")
            
            plt.show()
            
            return correlation_matrix
    
    return None

# Ejecutar visualizaciones si hay resultados disponibles
if 'df_results' in locals() and 'best_result' in locals():
    print("📊 Creando visualizaciones...")
    
    # Crear visualizaciones principales
    fig = create_tuning_visualizations(df_results, best_result)
    
    # Crear heatmap de correlación
    corr_matrix = create_hyperparameter_heatmap(df_results)
    
    print("✅ Visualizaciones completadas!")
    
else:
    print("⚠️  No hay resultados disponibles para visualización.")
    print("   Ejecuta primero las celdas de tuning y análisis.")

In [ ]:
# ================================
# ENTRENAMIENTO CON MEJOR CONFIGURACIÓN
# ================================

def train_with_best_config(best_config, full_training=False):
    """Entrena un modelo usando la mejor configuración encontrada."""
    
    print("🎯 ENTRENAMIENTO CON MEJOR CONFIGURACIÓN")
    print("=" * 50)
    
    # Crear configuración completa
    final_config = base_config.copy()
    
    # Actualizar con los mejores hiperparámetros
    sac_params = final_config.get('sac_params', {}).copy()
    
    param_mapping = {
        'lr': 'lr',
        'tau': 'tau', 
        'gamma': 'gamma',
        'alpha_lr': 'alpha_lr',
        'train_batch_size': 'train_batch_size',
        'replay_buffer_size': 'replay_buffer_size',
        'target_network_update_freq': 'target_network_update_freq',
        'num_steps_sampled_before_learning_starts': 'num_steps_sampled_before_learning_starts',
        'twin_q': 'twin_q',
        'clip_actions': 'clip_actions',
    }
    
    # Actualizar parámetros SAC
    for tune_param, sac_param in param_mapping.items():
        if tune_param in best_config:
            sac_params[sac_param] = best_config[tune_param]
    
    final_config['sac_params'] = sac_params
    
    # Configurar timesteps según el tipo de entrenamiento
    if full_training:
        final_config['training']['timesteps_total'] = 3_000_000  # Entrenamiento completo
        final_config['name'] = f"{base_config['name']}_OptimizedFull"
    else:
        final_config['training']['timesteps_total'] = 500_000  # Entrenamiento de validación
        final_config['name'] = f"{base_config['name']}_OptimizedDemo"
    
    # Actualizar path de almacenamiento
    final_config['storage_path'] = str(BASE_STORAGE_PATH.parent / 'optimized_models')
    
    print(f"📊 Configuración final:")
    print(f"  - Timesteps: {final_config['training']['timesteps_total']:,}")
    print(f"  - Nombre: {final_config['name']}")
    print(f"  - Almacenamiento: {final_config['storage_path']}")
    
    print(f"\n🎯 Hiperparámetros optimizados:")
    for param, value in best_config.items():
        print(f"  - {param}: {value}")
    
    return final_config

def run_optimized_training(final_config):
    """Ejecuta el entrenamiento con la configuración optimizada."""
    
    print(f"\n🚀 Iniciando entrenamiento optimizado...")
    print(f"⏱️  Esto puede tomar tiempo dependiendo de los timesteps configurados")
    
    # Importar función de entrenamiento de run.py
    from run import run_training
    
    try:
        # Ejecutar entrenamiento
        run_training(final_config)
        print("✅ Entrenamiento optimizado completado exitosamente!")
        
        # Guardar configuración utilizada
        config_file = Path(final_config['storage_path']) / f"{final_config['name']}_config.yaml"
        config_file.parent.mkdir(parents=True, exist_ok=True)
        
        with open(config_file, 'w') as f:
            yaml.dump(final_config, f, default_flow_style=False)
        
        print(f"💾 Configuración guardada en: {config_file}")
        
        return True
        
    except Exception as e:
        print(f"❌ Error durante el entrenamiento: {e}")
        return False

# Configurar opciones de entrenamiento
TRAINING_OPTIONS = {
    'demo': {
        'full_training': False,
        'description': 'Entrenamiento de demostración (500K timesteps)',
    },
    'full': {
        'full_training': True,
        'description': 'Entrenamiento completo (3M timesteps)',
    }
}

# Ejecutar entrenamiento con mejor configuración (si está disponible)
if 'best_result' in locals():
    print("🎯 Mejor configuración encontrada, preparando entrenamiento...")
    
    # Seleccionar tipo de entrenamiento
    training_type = 'demo'  # Cambiar a 'full' para entrenamiento completo
    
    print(f"🔧 Tipo de entrenamiento seleccionado: {training_type}")
    print(f"📝 Descripción: {TRAINING_OPTIONS[training_type]['description']}")
    
    # Preparar configuración
    final_config = train_with_best_config(
        best_result.config, 
        full_training=TRAINING_OPTIONS[training_type]['full_training']
    )
    
    # Preguntar confirmación (simulada)
    run_training_now = True  # Cambiar a False para evitar ejecución automática
    
    if run_training_now:
        print(f"\n🚀 Iniciando entrenamiento...")
        success = run_optimized_training(final_config)
        
        if success:
            print(f"\n🎉 ¡Entrenamiento completado exitosamente!")
            print(f"📁 Modelo guardado en: {final_config['storage_path']}")
            print(f"🎯 Nombre del modelo: {final_config['name']}")
        else:
            print(f"\n❌ El entrenamiento falló. Revisa los logs para más detalles.")
    else:
        print(f"\n⏸️  Entrenamiento preparado pero no ejecutado.")
        print(f"   Cambia 'run_training_now = True' para ejecutar.")
        
else:
    print("⚠️  No hay resultados de tuning disponibles.")
    print("   Ejecuta primero las celdas de tuning para encontrar la mejor configuración.")

In [ ]:
# ================================
# UTILIDADES PARA EXPERIMENTOS ANTERIORES
# ================================

def load_previous_experiment(experiment_name):
    """Carga resultados de un experimento anterior."""
    
    experiment_path = BASE_STORAGE_PATH / experiment_name
    
    if not experiment_path.exists():
        print(f"❌ No se encontró el experimento: {experiment_name}")
        return None
    
    # Buscar archivos de resultados
    results_file = experiment_path / f"{experiment_name}_results.csv"
    config_file = experiment_path / f"{experiment_name}_best_config.json"
    
    if results_file.exists():
        df_results = pd.read_csv(results_file)
        print(f"📊 Resultados cargados: {len(df_results)} trials")
        
        if config_file.exists():
            with open(config_file, 'r') as f:
                best_config = json.load(f)
            print(f"🎯 Mejor configuración cargada")
            
            return df_results, best_config
        else:
            print(f"⚠️  Archivo de configuración no encontrado: {config_file}")
            return df_results, None
    else:
        print(f"❌ No se encontraron resultados: {results_file}")
        return None

def list_available_experiments():
    """Lista los experimentos disponibles."""
    
    print("📋 EXPERIMENTOS DISPONIBLES")
    print("=" * 40)
    
    if not BASE_STORAGE_PATH.exists():
        print("❌ No hay experimentos disponibles")
        return []
    
    experiments = []
    for item in BASE_STORAGE_PATH.iterdir():
        if item.is_dir() and item.name.startswith('SAC_Tuning_'):
            experiments.append(item.name)
    
    if experiments:
        for i, exp in enumerate(experiments, 1):
            print(f"{i}. {exp}")
    else:
        print("❌ No hay experimentos de tuning disponibles")
    
    return experiments

def compare_experiments(experiment_names):
    """Compara múltiples experimentos."""
    
    print("🔍 COMPARACIÓN DE EXPERIMENTOS")
    print("=" * 50)
    
    comparison_data = []
    
    for exp_name in experiment_names:
        result = load_previous_experiment(exp_name)
        if result:
            df_results, best_config = result
            if df_results is not None:
                metric_col = f"last_{TUNING_CONFIG['metric'].replace('/', '_')}"
                
                if metric_col in df_results.columns:
                    best_score = df_results[metric_col].max() if TUNING_CONFIG['mode'] == 'max' else df_results[metric_col].min()
                    avg_score = df_results[metric_col].mean()
                    
                    comparison_data.append({
                        'experiment': exp_name,
                        'best_score': best_score,
                        'avg_score': avg_score,
                        'num_trials': len(df_results),
                        'success_rate': len(df_results[df_results['trial_status'] == 'TERMINATED']) / len(df_results) * 100
                    })
    
    if comparison_data:
        df_comparison = pd.DataFrame(comparison_data)
        print(df_comparison.to_string(index=False))
        
        # Crear visualización comparativa
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        
        # Comparar mejores scores
        axes[0].bar(df_comparison['experiment'], df_comparison['best_score'], color='skyblue', alpha=0.7)
        axes[0].set_title('Mejores Scores por Experimento')
        axes[0].set_ylabel(TUNING_CONFIG['metric'])
        axes[0].tick_params(axis='x', rotation=45)
        
        # Comparar scores promedio
        axes[1].bar(df_comparison['experiment'], df_comparison['avg_score'], color='lightgreen', alpha=0.7)
        axes[1].set_title('Scores Promedio por Experimento')
        axes[1].set_ylabel(TUNING_CONFIG['metric'])
        axes[1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        return df_comparison
    
    return None

# Utilidades interactivas
print("🛠️  UTILIDADES PARA EXPERIMENTOS")
print("=" * 40)

# Listar experimentos disponibles
available_experiments = list_available_experiments()

# Ejemplo de uso:
print(f"\n📖 EJEMPLOS DE USO:")
print(f"# Cargar experimento anterior:")
print(f"# df_results, best_config = load_previous_experiment('SAC_Tuning_sac_random_search_oval_small')")
print(f"")
print(f"# Comparar experimentos:")
print(f"# comparison = compare_experiments(['experiment1', 'experiment2'])")
print(f"")
print(f"# Usar configuración de experimento anterior:")
print(f"# final_config = train_with_best_config(best_config, full_training=True)")

if available_experiments:
    print(f"\n💡 Puedes cargar cualquiera de los experimentos listados arriba usando:")
    print(f"   load_previous_experiment('nombre_del_experimento')")

## 🚀 Guía de Uso

### 1. Configuración Inicial
- **Experimento base**: Cambia `TUNING_CONFIG['base_experiment']` para usar un experimento diferente como template
- **Número de muestras**: Ajusta `TUNING_CONFIG['num_samples']` para controlar cuántas configuraciones probar
- **Timesteps por trial**: Modifica `TUNING_CONFIG['max_timesteps']` para balancear tiempo vs calidad

### 2. Espacios de Búsqueda
- **`basic`**: Hiperparámetros fundamentales de SAC
- **`extended`**: Incluye parámetros adicionales para búsqueda más exhaustiva
- **`grid`**: Búsqueda sistemática en una grid predefinida

### 3. Algoritmos de Búsqueda
- **`random`**: Búsqueda aleatoria (rápida, buena baseline)
- **`bayesian`**: Optimización Bayesiana (más inteligente, mejor para pocas muestras)
- **`hyperopt`**: Tree-structured Parzen Estimator (TPE)

### 4. Schedulers
- **`asha`**: Async Successive Halving (early stopping eficiente)
- **`pbt`**: Population Based Training (evolución de hiperparámetros)
- **`plateau`**: Para por convergencia

### 5. Monitoreo
- **Ray Dashboard**: http://localhost:8265 (mientras Ray está corriendo)
- **Archivos de log**: Guardados en `BASE_STORAGE_PATH`
- **Visualizaciones**: Generadas automáticamente después del tuning

### 6. Mejores Prácticas
1. **Empezar pequeño**: Usa pocas muestras y timesteps bajos para probar
2. **Monitorear recursos**: El tuning puede ser computacionalmente intensivo
3. **Guardar configuraciones**: Siempre guarda las mejores configuraciones encontradas
4. **Comparar experimentos**: Usa las utilidades para comparar diferentes enfoques

### 7. Configuraciones Recomendadas

#### Para experimentación rápida:
```python
TUNING_CONFIG['num_samples'] = 10
TUNING_CONFIG['max_timesteps'] = 100_000
SELECTED_SEARCH_SPACE = 'basic'
SELECTED_EXPERIMENT = 'sac_random_search'
```

#### Para búsqueda exhaustiva:
```python
TUNING_CONFIG['num_samples'] = 50
TUNING_CONFIG['max_timesteps'] = 1_000_000
SELECTED_SEARCH_SPACE = 'extended'
SELECTED_EXPERIMENT = 'sac_bayesian_optimization'
```

#### Para búsqueda sistemática:
```python
SELECTED_SEARCH_SPACE = 'grid'
SELECTED_EXPERIMENT = 'sac_random_search'
# num_samples se ignora en grid search
```

## 📋 Resumen del Notebook

Este notebook implementa un sistema completo de hyperparameter tuning para SAC que:

✅ **Reutiliza código existente** de `run.py` sin modificaciones  
✅ **Implementa múltiples algoritmos** de búsqueda (Random, Bayesian, HyperOpt)  
✅ **Incluye early stopping** para eficiencia computacional  
✅ **Genera visualizaciones** automáticas de resultados  
✅ **Guarda configuraciones** óptimas para uso posterior  
✅ **Permite comparación** entre diferentes experimentos  

## 🎯 Siguientes Pasos

1. **Ejecutar tuning básico**: Comienza con configuración `basic` y pocas muestras
2. **Analizar resultados**: Revisa visualizaciones y métricas
3. **Refinar búsqueda**: Usa mejores configuraciones como punto de partida
4. **Entrenamiento completo**: Usa la mejor configuración para entrenamiento final
5. **Evaluar modelo**: Usa las utilidades de evaluación de `run.py`

## 🔧 Personalización

Para adaptar este notebook a tus necesidades específicas:

- **Cambiar métricas**: Modifica `TUNING_CONFIG['metric']` 
- **Añadir parámetros**: Extiende `search_spaces` con nuevos hiperparámetros
- **Cambiar mapas**: Modifica `base_experiment` para usar diferentes mapas
- **Ajustar recursos**: Cambia `max_concurrent_trials` según tu hardware

## 📚 Referencias

- [Ray Tune Documentation](https://docs.ray.io/en/latest/tune/index.html)
- [SAC Algorithm Paper](https://arxiv.org/abs/1801.01290)
- [Hyperparameter Optimization Best Practices](https://docs.ray.io/en/latest/tune/tutorials/tune-stopping.html)

---

**¡Listo para optimizar tus algoritmos SAC! 🚀**